# RAG sobre datos personales y el desarrollo de la IA

## Documentos

Las fuentes, fechas y sumas de verificación están en [`data/manifest.json`](../data/manifest.json).

| Documento | Tipo | Estatus | Páginas | Caracteres | Tablas |
|---|---|---|---:|---:|---:|
| LFPDPPP | Ley | Vigente; publicada en 2025, última reforma el 14-11-2025 | 24 | 66,650 | 0 |
| LGPDPPSO | Ley | Vigente; publicada en 2025, última reforma el 14-11-2025 | 37 | 136,701 | 0 |
| Reglamento de la LFPDPPP | Reglamento | Publicado como texto vigente; redactado para la ley de 2010 | 36 | 102,994 | 0 |
| Recomendaciones del INAI sobre datos personales e IA | Guía | No vinculante; edición de junio de 2024 | 48 | 93,635 | 26 |
| Evaluación de preparación en IA de UNESCO para México | Informe | No vinculante; 2024 | 74 | 201,353 | 2 |
| Principios de Chapultepec | Declaración | No vinculante; enero de 2026 | 2 | 7,425 | 0 |
| Iniciativa de Ley General para el Uso y Control de la IA | Iniciativa | Propuesta; no es ley vigente | 42 | 82,119 | 0 |
| **Total** | | | **263** | **690,877** | **28** |

Con fragmentos de 1,000 caracteres y traslape de 150, el corpus daría unos 800 chunks. Es solo una estimación: la estrategia de chunking aún no está definida.

## Por qué estos documentos

- **Tema común con traslape real.** Los derechos ARCO (_Acceso, Rectificación, Cancelación, Oposición_) aparecen en las dos leyes y el reglamento (17, 46 y 22 menciones), y el consentimiento aparece en seis de los siete documentos. La IA aparece en cuatro. Así, el retriever tiene pasajes plausibles pero incorrectos entre los cuales elegir.

- **Contraste entre estrategias de retrieval.** Hay términos exactos que deberían favorecer a BM25 (`ARCO`, `LFPDPPP`, números de artículo) y conceptos que deberían favorecer a los embeddings, como pedir a una empresa que borre tus datos.
- **Confusión difícil.** La LFPDPPP y la LGPDPPSO comparten mucho vocabulario, pero una regula a particulares y la otra a sujetos obligados del sector público.
- **Autoridad distinta.** La mezcla de ley, guía y propuesta permite evaluar si el sistema presenta como vigente algo que no lo es.

## Chunking

La entrada es `data/elements.jsonl`, que genera `parser.ipynb`: un registro por elemento con todas las decisiones de parsing aplicadas.

- **Flujo de lectura:** elementos con `removed` vacío, `is_note` falso y texto.
- **Notas al pie:** elementos con `removed` vacío e `is_note` verdadero.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

DATA = Path("../data")
MANIFEST = {entry["doc_id"]: entry for entry in json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))["documents"]}

elements = pd.read_json(DATA / "elements.jsonl", lines=True)
kept = elements["removed"].isna()
flow = elements[kept & ~elements["is_note"] & (elements["text"].str.strip() != "")]
notes = elements[kept & elements["is_note"]]
print(f"Flujo de lectura: {len(flow)} elementos, {flow['text'].str.len().sum():,} caracteres")
print(f"Notas al pie: {len(notes)}, {notes['text'].str.len().sum():,} caracteres")

Flujo de lectura: 3279 elementos, 682,949 caracteres
Notas al pie: 288, 46,318 caracteres


## Decisión: estrategia de chunking

Los chunks se forman juntando elementos consecutivos del flujo de lectura hasta un límite de tamaño:

- Un título de sección empieza un chunk nuevo. Los títulos seguidos («Capítulo I», «Disposiciones generales») se quedan juntos y con el texto que les sigue.
- En las leyes, el reglamento y la iniciativa, cada «Artículo N» también empieza un chunk nuevo, así que un chunk no mezcla artículos.
- Los elementos no se cortan, salvo los que son más largos que el límite (ver «Decisión: tablas largas y chunks pequeños»).
- Cada chunk guarda sus páginas del PDF, a partir de las de sus elementos, para compararlo con las respuestas esperadas.

Comparación con el límite final de 1,500 caracteres (ver «Decisión: límite de tamaño y traslape»), calculada en la celda que sigue al código del chunking:

| | Tamaño fijo (1,500 caracteres, traslape de 225) | Elementos agrupados |
|---|---:|---:|
| Chunks | 542 | 865 |
| Empiezan a mitad de un elemento | 533 | 0 |
| Chunks legales que mezclan texto de dos o más artículos | 220 de 305 | 0 |

- **Tamaño fijo:** se simula sobre el mismo texto, con ventanas de 1,500 caracteres y un traslape de 225 (15 %, una proporción habitual).
- **Empieza a mitad de un elemento:** el inicio del chunk no coincide con el inicio de un elemento de Docling (párrafo, elemento de lista, título o tabla).
- **Mezcla artículos:** el chunk tiene texto de dos o más artículos. Solo se cuenta en las leyes, el reglamento y la iniciativa.
- Con elementos agrupados, los dos conteos son 0 por construcción, y la celda lo confirma. La excepción son las 11 tablas más largas que el límite, que se dividen entre filas (ver «Decisión: tablas largas y chunks pequeños»).
- Los elementos agrupados dan más chunks, y más pequeños, porque la mayoría de los artículos son cortos.

Se descartaron:

- **Tamaño fijo:** corta oraciones y artículos.
- **División recursiva** (párrafo, luego oración): sigue mezclando artículos e ignora los títulos.
- **Chunking semántico:** necesita el modelo de embeddings antes de dividir y es más difícil de explicar y reproducir, cuando los documentos ya marcan su estructura.
- **El chunker de Docling:** trabaja sobre el documento original de Docling, así que ignoraría la limpieza del parsing.

In [2]:
HEADING_LABELS = {"section_header", "title"}
ARTICLE_TYPES = {"ley", "reglamento", "iniciativa de ley"}
ARTICLE_START = re.compile(r"(Art[íi]culo\s+\d+)", re.IGNORECASE)


def starts_chunk(element, doc_type):
    """A section heading, or an article in a legal text, starts a new chunk."""
    if element["label"] in HEADING_LABELS:
        return True
    return doc_type in ARTICLE_TYPES and bool(ARTICLE_START.match(element["text"]))


def current_articles(doc_flow):
    """Article in force at each element: set by an "Artículo N" element and reset by any other heading."""
    articles, current = [], None
    for label, text in zip(doc_flow["label"], doc_flow["text"]):
        match = ARTICLE_START.match(text)
        if match:
            current = match.group(1)
        elif label in HEADING_LABELS:
            current = None
        articles.append(current)
    return articles


def pack_elements(doc_flow, doc_type, max_chars):
    """Group consecutive elements into chunks of up to max_chars characters without cutting an element.
    A chunk that only holds headings stays open, so headings stay with the text that follows them."""
    groups, current = [], []
    for element in doc_flow.to_dict("records"):
        size = sum(len(item["text"]) for item in current)
        has_body = any(item["label"] not in HEADING_LABELS for item in current)
        if current and has_body and (starts_chunk(element, doc_type) or size + len(element["text"]) > max_chars):
            groups.append(current)
            current = []
        current.append(element)
    if current:
        groups.append(current)
    return groups


def chunk_record(doc_id, number, group, doc_type):
    """One chunk. `continues` marks a chunk whose first element was moved to it by the size limit,
    so it carries on an article or section that started in the previous chunk."""
    body = [item for item in group if item["label"] not in HEADING_LABELS] or group
    pages = sorted({page for item in group for page in range(int(item["page"]), int(item["page_end"]) + 1)})
    return {
        "chunk_id": f"{doc_id}-{number:04d}",
        "doc_id": doc_id,
        "pages": pages,
        "heading": body[0]["heading"],
        "n_elements": len(group),
        "continues": number > 1 and not starts_chunk(group[0], doc_type),
        "article": group[0]["article"],
        "text": "\n\n".join(item["text"] for item in group),
    }


def build_chunks(max_chars):
    records = []
    for doc_id, doc_flow in flow.groupby("doc_id", sort=False):
        doc_type = MANIFEST[doc_id]["type"]
        articles = current_articles(doc_flow) if doc_type in ARTICLE_TYPES else [None] * len(doc_flow)
        groups = pack_elements(doc_flow.assign(article=articles), doc_type, max_chars)
        records += [chunk_record(doc_id, number, group, doc_type) for number, group in enumerate(groups, start=1)]
    return pd.DataFrame(records)

In [3]:
# Comparación con el corte de tamaño fijo, con el mismo límite de 1,500 caracteres
COMPARE_CHARS = 1500
COMPARE_OVERLAP = 225  # 15 % del tamaño, la proporción habitual de traslape


def fixed_windows(texts, articles, size, overlap):
    """Corte de tamaño fijo con traslape sobre el mismo texto que usan los chunks (elementos unidos con un renglón en blanco).
    Para cada ventana: si empieza a mitad de un elemento y cuántos artículos distintos tienen texto dentro de ella."""
    starts, position = [], 0
    for text in texts:
        starts.append(position)
        position += len(text) + 2
    total, element_starts = position - 2, set(starts)
    windows = []
    for start in range(0, total, size - overlap):
        end = start + size
        inside = {article for first, text, article in zip(starts, texts, articles) if article and first < end and first + len(text) > start}
        windows.append({"mid": start not in element_starts, "articles": len(inside)})
        if end >= total:
            break
    return windows


rows = []
for doc_id, doc_flow in flow.groupby("doc_id", sort=False):
    doc_type = MANIFEST[doc_id]["type"]
    articles = current_articles(doc_flow) if doc_type in ARTICLE_TYPES else [None] * len(doc_flow)
    windows = fixed_windows(doc_flow["text"].tolist(), articles, COMPARE_CHARS, COMPARE_OVERLAP)
    groups = pack_elements(doc_flow.assign(article=articles), doc_type, COMPARE_CHARS)
    rows.append({
        "documento": doc_id,
        "fijo: chunks": len(windows),
        "fijo: empiezan a mitad de un elemento": sum(window["mid"] for window in windows),
        "fijo: mezclan artículos": sum(window["articles"] >= 2 for window in windows),
        "agrupados: chunks": len(groups),
        "agrupados: mezclan artículos": sum(len({item["article"] for item in group if isinstance(item["article"], str)}) >= 2 for group in groups),
    })
comparison = pd.DataFrame(rows).set_index("documento")
comparison.loc["Total"] = comparison.sum()
comparison

,fijo: chunks,fijo: empiezan a mitad de un elemento,fijo: mezclan artículos,agrupados: chunks,agrupados: mezclan artículos
documento,,,,,
lfpdppp,52,50,32,89,0
lgpdppso,107,106,73,185,0
reglamento_lfpdppp,81,80,71,161,0
inai_recomendaciones_ia,73,72,0,85,0
unesco_ram_mexico,158,156,0,177,0
principios_chapultepec,6,5,0,11,0
iniciativa_ley_ia_senado,65,64,44,157,0
Total,542,533,220,865,0


## Decisión: límite de tamaño y traslape

La tabla mide los chunks con distintos límites. Los tokens se estiman con 4 caracteres por token.

In [4]:
def chunk_stats(max_chars):
    chunks = build_chunks(max_chars)
    sizes = chunks["text"].str.len()
    legal = chunks["doc_id"].map(lambda doc_id: MANIFEST[doc_id]["type"] in ARTICLE_TYPES)
    return {
        "límite": max_chars,
        "≈ tokens": max_chars // 4,
        "chunks": len(chunks),
        "mediana": int(sizes.median()),
        "p90": int(sizes.quantile(0.9)),
        "continúan un artículo o sección": int(chunks["continues"].sum()),
        "de ellos, en textos legales": int((chunks["continues"] & legal).sum()),
        "elementos más largos que el límite": int((flow["text"].str.len() > max_chars).sum()),
    }


pd.DataFrame([chunk_stats(limit) for limit in [500, 800, 1000, 1500, 2000, 3000]]).set_index("límite")

,≈ tokens,chunks,mediana,p90,continúan un artículo o sección,"de ellos, en textos legales",elementos más largos que el límite
límite,,,,,,,
500,125,1717,389,543,1113,570,210
800,200,1253,560,775,649,278,43
1000,250,1080,651,967,476,184,20
1500,375,865,702,1429,261,87,11
2000,500,768,665,1895,164,46,6
3000,750,695,593,2712,91,22,1


Se usa un límite de **1,500 caracteres (unos 375 tokens) sin traslape**.

- **Cabe en los modelos de embeddings que leen hasta 512 tokens**, con margen; 2,000 caracteres en español pueden pasar ese límite.
- **Parte la mitad de artículos que 1,000 caracteres** (87 chunks legales que continúan un artículo, frente a 184), y solo 11 elementos son más largos que el límite, todos tablas.
- **La mediana casi no cambia por encima de 1,000**, porque la mayoría de los artículos son cortos: un límite mayor solo reduce cuántas veces se parte un artículo largo.

El límite cuenta los caracteres de los elementos; los saltos de línea que los separan dentro del chunk pueden sumar unos pocos más.

**Sin traslape**, porque los elementos no se cortan: no se pierde ninguna oración entre un chunk y el siguiente. El problema de los chunks que continúan un artículo es otro: no dicen de qué artículo son (por ejemplo, las fracciones XII a XX del artículo 3). Eso se resuelve con el contexto al inicio de cada chunk, que es la siguiente decisión.

In [5]:
MAX_CHARS = 1500
chunks = build_chunks(MAX_CHARS)
sizes = chunks["text"].str.len()
print(f"{len(chunks)} chunks; mediana de {int(sizes.median())} caracteres; {chunks['continues'].sum()} continúan un artículo o sección")
chunks.groupby("doc_id", sort=False).agg(
    chunks=("chunk_id", "size"),
    mediana=("text", lambda texts: int(texts.str.len().median())),
    continúan=("continues", "sum"),
    más_de_una_página=("pages", lambda pages: sum(len(p) > 1 for p in pages)),
)

865 chunks; mediana de 702 caracteres; 261 continúan un artículo o sección


,chunks,mediana,continúan,más_de_una_página
doc_id,,,,
lfpdppp,89,629,19,14
lgpdppso,185,637,42,28
reglamento_lfpdppp,161,512,15,27
inai_recomendaciones_ia,85,1147,48,13
unesco_ram_mexico,177,1268,124,34
principios_chapultepec,11,477,2,1
iniciativa_ley_ia_senado,157,397,11,24


## Decisión: contexto al inicio de cada chunk

Cada chunk empieza con una línea de contexto: el título del documento, el título de sección vigente y, si el chunk continúa un artículo, el número del artículo.

```
Ley Federal de Protección de Datos Personales en Posesión de los Particulares (LFPDPPP) | Disposiciones Generales | Artículo 2 (continuación)
```

Por qué:

- **Los números de artículo no identifican la ley.** 137 de los 144 números de artículo aparecen en dos o más de los cuatro textos legales, y 62 en los cuatro.
- **Los chunks casi nunca nombran su documento.** Solo 84 de 865 lo hacen, así que una pregunta como «¿qué dice la LFPDPPP sobre…?» no encontraría los chunks de la propia ley por su nombre.
- **Los chunks que continúan un artículo no dicen cuál es.** De los 87 chunks legales que continúan un artículo o sección, 82 no contienen «Artículo N».

Las dos leyes y el reglamento llevan su sigla después del título, porque así suelen citarse. El artículo vigente lo fija cada «Artículo N» y lo reinicia cualquier otro título (por ejemplo, «Transitorios»); se agrega solo si el título de sección no lo dice ya. Así, 53 chunks reciben su artículo; de los otros 34 que continúan, 5 ya lo tienen en el título de sección y 29 son de los artículos transitorios («Tercero.-», «Quinto.-»), que no llevan número de artículo y quedan con «Transitorios» como título.

La línea de contexto va en el texto que se usa para buscar y en el que lee el LLM (`full_text`); el texto original queda en `text`. Se descartaron no agregar contexto, agregar solo título y sección (los chunks que continúan seguirían sin artículo) y pedir a un LLM que escriba el contexto de cada chunk (865 llamadas con costo y resultados no reproducibles; queda como mejora posible en producción).

In [6]:
SHORT_NAMES = {"lfpdppp": "LFPDPPP", "lgpdppso": "LGPDPPSO", "reglamento_lfpdppp": "Reglamento de la LFPDPPP"}


def context_line(chunk):
    title = MANIFEST[chunk["doc_id"]]["title"]
    if chunk["doc_id"] in SHORT_NAMES:
        title += f" ({SHORT_NAMES[chunk['doc_id']]})"
    parts = [title]
    heading = chunk["heading"] if isinstance(chunk["heading"], str) else ""
    article = chunk["article"] if isinstance(chunk["article"], str) else ""
    if heading:
        parts.append(heading)
    if chunk["continues"] and article and not heading.startswith(article):
        parts.append(f"{article} (continuación)")
    return " | ".join(parts)


chunks["context"] = chunks.apply(context_line, axis=1)
chunks["full_text"] = chunks["context"] + "\n\n" + chunks["text"]

with_article = chunks["context"].str.endswith("(continuación)")
print(f"Chunks con el artículo en el contexto: {with_article.sum()}")
print(f"Largo de la línea de contexto: mediana {int(chunks['context'].str.len().median())}, máximo {chunks['context'].str.len().max()} caracteres")
print(f"Largo de full_text: máximo {chunks['full_text'].str.len().max()} caracteres; "
      f"{(chunks['full_text'].str.len() > 2000).sum()} chunks pasan de 2,000")
for chunk_id in ["lfpdppp-0006", "lfpdppp-0046", "inai_recomendaciones_ia-0010"]:
    print(f"\n{chunk_id}: {chunks.set_index('chunk_id').at[chunk_id, 'context']}")

Chunks con el artículo en el contexto: 53
Largo de la línea de contexto: mediana 147, máximo 283 caracteres
Largo de full_text: máximo 4700 caracteres; 8 chunks pasan de 2,000

lfpdppp-0006: Ley Federal de Protección de Datos Personales en Posesión de los Particulares (LFPDPPP) | Disposiciones Generales | Artículo 2 (continuación)

lfpdppp-0046: Ley Federal de Protección de Datos Personales en Posesión de los Particulares (LFPDPPP) | Artículo 39. La Secretaría tiene las siguientes atribuciones:

inai_recomendaciones_ia-0010: Recomendaciones para el tratamiento de datos personales derivado del uso de la inteligencia artificial | INTRODUCCIÓN


## Decisión: notas al pie en el índice

Solo el INAI (90 notas) y UNESCO (198) tienen notas al pie. 84 son solo un enlace («Disponible en: URL [consulta: fecha]»): 77 del INAI y 7 de UNESCO. Las demás son sobre todo referencias (autor, año, título y enlace), pero unas 56 traen explicaciones o definiciones, como la de inteligencia artificial (nota 1 de UNESCO) o las de red neuronal, reconocimiento facial y cómputo en la nube (notas 28, 53 y 76 del INAI).

Se indexan las notas con texto, agrupadas por página:

- Una nota es «solo un enlace» si, al quitarle su número, los enlaces y las frases «Disponible en» y «[consulta: …]», no le quedan más de 5 caracteres. Esas no se indexan.
- Las notas de una misma página del PDF forman un chunk, hasta el mismo límite de 1,500 caracteres, sin cortar ninguna nota.
- Su línea de contexto es el título del documento y «Notas al pie, página N» (página del PDF).
- La columna `kind` distingue los chunks de texto (`texto`) de los de notas (`notas`).

Se descartaron no indexarlas (se perderían las definiciones), indexar también las que solo son un enlace (solo agregan URLs, y la búsqueda por palabras clave puede coincidir con palabras dentro de ellas) y hacer un chunk por nota (204 chunks muy cortos, que pueden quedar arriba solo por coincidir en una o dos palabras). La desventaja de agrupar por página es que las notas de una página tratan temas distintos.

In [7]:
LINK_PARTS = re.compile(r"https?://\S+|www\.\S+|Disponible en:?|\[?consulta:?[^\]]*\]?|Obtenido de", re.IGNORECASE)


def is_link_only(text):
    """True when almost nothing is left of a note after removing its number, its links and the citation phrases."""
    rest = LINK_PARTS.sub("", re.sub(r"^\d+\s+", "", text))
    return len(re.sub(r"[\s.,;:\[\]]+", " ", rest).strip()) <= 5


def note_record(doc_id, number, page, texts):
    return {
        "chunk_id": f"{doc_id}-nota-{number:03d}",
        "doc_id": doc_id,
        "pages": [int(page)],
        "heading": f"Notas al pie, página {int(page)}",
        "n_elements": len(texts),
        "continues": False,
        "article": None,
        "text": "\n\n".join(texts),
    }


def build_note_chunks(text_notes, max_chars):
    """Group the notes of each PDF page into chunks of up to max_chars characters, without cutting a note."""
    records = []
    for doc_id, doc_notes in text_notes.groupby("doc_id", sort=False):
        number = 0
        for page, page_notes in doc_notes.groupby("page"):
            group, size = [], 0
            for text in page_notes["text"]:
                if group and size + len(text) > max_chars:
                    number += 1
                    records.append(note_record(doc_id, number, page, group))
                    group, size = [], 0
                group.append(text)
                size += len(text)
            number += 1
            records.append(note_record(doc_id, number, page, group))
    return pd.DataFrame(records)


link_only = notes["text"].map(is_link_only)
note_chunks = build_note_chunks(notes[~link_only], MAX_CHARS)
note_chunks["context"] = note_chunks.apply(context_line, axis=1)
note_chunks["full_text"] = note_chunks["context"] + "\n\n" + note_chunks["text"]

text_chunks = chunks[~chunks["chunk_id"].str.contains("-nota-")]
chunks = pd.concat([text_chunks.assign(kind="texto"), note_chunks.assign(kind="notas")], ignore_index=True)

print(f"Notas: {len(notes)}; solo enlace: {link_only.sum()}; indexadas: {(~link_only).sum()}")
print(f"Chunks de notas: {len(note_chunks)}; mediana de {int(note_chunks['text'].str.len().median())} caracteres")
print(f"Chunks en total: {len(chunks)}")
print(f"\nEjemplo: {note_chunks.iloc[0]['context']}")
chunks.groupby(["doc_id", "kind"], sort=False).size().unstack(fill_value=0)

Notas: 288; solo enlace: 84; indexadas: 204
Chunks de notas: 49; mediana de 628 caracteres
Chunks en total: 914

Ejemplo: Recomendaciones para el tratamiento de datos personales derivado del uso de la inteligencia artificial | Notas al pie, página 11


kind,texto,notas
doc_id,,
lfpdppp,89,0
lgpdppso,185,0
reglamento_lfpdppp,161,0
inai_recomendaciones_ia,85,10
unesco_ram_mexico,177,39
principios_chapultepec,11,0
iniciativa_ley_ia_senado,157,0


## Decisión: tablas largas y chunks pequeños

**Tablas largas.** 11 elementos son más largos que el límite de 1,500 caracteres, y todos son tablas. La más larga es la Tabla 2 de UNESCO (p. 52, 4,564 caracteres). Muchos modelos de embeddings solo leen los primeros 512 tokens, así que el final de esas tablas no se vería al buscar.

Los chunks que tienen un elemento más largo que el límite se dividen:

- Primero entre filas de la tabla. Cada fila ya lleva el nombre de sus columnas, así que se entiende sola.
- Si una fila sigue siendo más larga que el límite (las tablas de una sola fila de las pp. 38 y 40 del INAI), se divide al final de una oración, sin separar un número de lista («4.») del texto que le sigue.
- Cada pieza conserva el contexto, las páginas y el título del chunk original. La primera conserva su identificador y las demás llevan `-2`, `-3`…

Se descartaron dejarlas enteras y dividir solo entre filas, porque las dos tablas de una sola fila seguirían pasando el límite.

**Chunks pequeños.** 44 chunks de texto tienen menos de 200 caracteres. 22 son artículos cortos completos, unos 20 son un título con un texto o elemento de lista corto, y uno es la contraportada de UNESCO («Síguenos @UNESCO #AI #HumanAI»). Al dividir las tablas aparecen 3 más, porque el final de una tabla puede quedar corto (por ejemplo, la última fila de la tabla de la p. 12 del INAI). Todos se quedan como están: la línea de contexto ya les agrega unos 150 caracteres, y unirlos al chunk siguiente mezclaría artículos, que es lo que la estrategia evita.

In [8]:
SENTENCE_BREAK = re.compile(r"(?<=[.;:!?])\s+")
LIST_MARK = re.compile(r"\(?(\d{1,3}|[IVX]{1,4}|[a-z])[.)]")


def sentences(row):
    """Split a row at sentence ends, keeping a list number such as "4." with the text that follows it."""
    merged = []
    for part in SENTENCE_BREAK.split(row):
        if merged and LIST_MARK.fullmatch(merged[-1]):
            merged[-1] += " " + part
        else:
            merged.append(part)
    return merged


def text_units(text, max_chars):
    """(separator, unit) pairs: the chunk's elements, and rows and sentences only inside elements longer than max_chars."""
    units = []
    for part in text.split("\n\n"):
        if len(part) <= max_chars:
            units.append(("\n\n", part))
            continue
        for row_number, row in enumerate(part.split("\n")):
            row_units = [row] if len(row) <= max_chars else sentences(row)
            for sentence_number, sentence in enumerate(row_units):
                if sentence_number > 0:
                    separator = " "
                elif row_number > 0:
                    separator = "\n"
                else:
                    separator = "\n\n"
                units.append((separator, sentence))
    return units


def split_text(text, max_chars):
    """Pieces of up to max_chars characters, cut only inside elements longer than the limit."""
    pieces, current = [], ""
    for separator, unit in text_units(text, max_chars):
        if current and len(current) + len(separator) + len(unit) > max_chars:
            pieces.append(current)
            current = unit
        else:
            current = current + separator + unit if current else unit
    if current:
        pieces.append(current)
    return pieces


def split_long_chunks(chunks, max_chars):
    records = []
    for chunk in chunks.to_dict("records"):
        has_long_element = any(len(part) > max_chars for part in chunk["text"].split("\n\n"))
        pieces = split_text(chunk["text"], max_chars) if has_long_element else [chunk["text"]]
        for number, piece in enumerate(pieces, start=1):
            record = dict(chunk, text=piece)
            if number > 1:
                record.update(chunk_id=f"{chunk['chunk_id']}-{number}", continues=True)
            record["context"] = context_line(record)
            record["full_text"] = record["context"] + "\n\n" + piece
            records.append(record)
    return pd.DataFrame(records)


before = len(chunks)
long_ids = chunks.loc[chunks["text"].map(lambda text: any(len(part) > MAX_CHARS for part in text.split("\n\n"))), "chunk_id"]
chunks = split_long_chunks(chunks, MAX_CHARS)
sizes = chunks["text"].str.len()
print(f"Chunks con una tabla más larga que el límite: {len(long_ids)}; chunks: {before} antes, {len(chunks)} después")
print(f"Texto más largo: {sizes.max()} caracteres; full_text más largo: {chunks['full_text'].str.len().max()}")
print(f"Chunks de texto con menos de 200 caracteres (se quedan): {((chunks['kind'] == 'texto') & (sizes < 200)).sum()}")
print("\nTabla 2 de UNESCO (p. 52):")
for _, chunk in chunks[chunks["chunk_id"].str.startswith(long_ids[long_ids.str.startswith("unesco")].iloc[-1])].iterrows():
    print(f"  {chunk['chunk_id']}: {len(chunk['text'])} caracteres, empieza con {chunk['text'][:70]!r}")

Chunks con una tabla más larga que el límite: 11; chunks: 914 antes, 927 después
Texto más largo: 1523 caracteres; full_text más largo: 1734
Chunks de texto con menos de 200 caracteres (se quedan): 47

Tabla 2 de UNESCO (p. 52):
  unesco_ram_mexico-0112: 1443 caracteres, empieza con 'RECOMENDACIONES PRINCIPALES\n\nRUBRO: 1. MAPA DEL ECOSISTEMA DE LA IA; R'
  unesco_ram_mexico-0112-2: 1138 caracteres, empieza con 'RUBRO: 4. EMITIR LA ESTRATEGIA NACIONAL DE IA; RECOMENDACIONES: 4. EMI'
  unesco_ram_mexico-0112-3: 775 caracteres, empieza con 'RUBRO: 4.2 ÉTICA, RESPONSABILIDAD E INCLUSIÓN. IA DE INTERÉS PÚBLICO; '
  unesco_ram_mexico-0112-4: 1234 caracteres, empieza con 'RUBRO: 4.3 ACTUALIZACIÓN EDUCATIVA Y RECONVERSIÓN LABORAL; RECOMENDACI'


## Consultas de evaluación

### Decisión: tipo de preguntas

Las 10 consultas son preguntas realistas: las que haría alguien que usa el sistema, por ejemplo un responsable de cumplimiento o un desarrollador que trabaja con datos personales. No se escogen para probar un tipo de contenido en particular. Al menos 2 son preguntas trampa, como pide el enunciado, redactadas también como las haría un usuario.

Se descartaron:

- **Preguntas dirigidas a los retos encontrados en el parsing y el chunking** (un artículo pedido por número y sigla, una definición dicha con otras palabras, una cifra de una tabla, una definición que solo está en una nota al pie…). Darían más material para el análisis de errores, pero no representan cómo se usaría el sistema.
- **Preguntas escritas a partir de pasajes elegidos al azar.** No tienen sesgo, pero suelen ser triviales y algunos documentos podrían quedarse sin preguntas.

La consecuencia es que puede que ninguna pregunta toque las tablas, las notas al pie o el texto escaneado de la iniciativa. El análisis de errores cubre lo que las preguntas sí tocan.

### Decisión: cómo se anotan las respuestas esperadas

Cada consulta se anota en `data/queries.json` con su pregunta, su respuesta ideal y los pasajes que la responden. Cada pasaje lleva el documento, la página del PDF y la cita exacta: un párrafo o una fila de tabla (o parte de ellos) copiados de `data/markdown/`.

```json
{
  "id": "q01",
  "question": "…",
  "type": "…",
  "ideal_answer": "…",
  "evidence": [{"doc_id": "lfpdppp", "page": 5, "quote": "…"}]
}
```

Los chunks esperados de una consulta son los de ese documento cuyo texto contiene alguna de sus citas (sin contar diferencias de espacios y saltos de línea). Una cita que no aparece en ningún chunk se avisa, para corregirla: puede ser una nota que solo es un enlace (no se indexan) o un pasaje que cruza de un chunk a otro.

Así, la anotación no depende de los identificadores de los chunks: si la segmentación cambia, las citas vuelven a encontrar los chunks correctos. Se descartaron anotar los identificadores de los chunks (habría que rehacer la anotación si cambian) y anotar solo documento y página (una página de una ley tiene 3 o 4 chunks, así que un chunk de la página correcta pero de otro artículo contaría como correcto).

Al probarlo con 105 pasajes copiados al azar de `data/markdown/`, todos aparecieron en algún chunk, salvo 4 notas al pie que solo son un enlace.

### Las 10 consultas

Las propuso el asistente de IA usado en el proyecto, a partir de candidatas cuyas respuestas se verificaron contra los chunks. Son 8 respondibles y 2 trampas, y entre todas cubren los 7 documentos:

- **q09, premisa falsa:** pregunta por las multas de una «ley de inteligencia artificial vigente», que no existe; solo hay una iniciativa en el Senado.
- **q10, premisa desactualizada:** pregunta cómo quejarse «ante el INAI», cuyas funciones pasaron a la Secretaría Anticorrupción y Buen Gobierno con la ley de 2025.

Las trampas también tienen pasajes esperados (los que corrigen la premisa), así que entran en Precision@k y Recall@k como las demás. En la generación se revisa además si la respuesta corrige la premisa.

Riesgo: quien redactó las preguntas leyó el corpus, así que algunas pueden repetir su vocabulario y favorecer a la búsqueda por palabras clave.

In [9]:
from IPython.display import display

QUERIES_FILE = DATA / "queries.json"


def normalize(text):
    return re.sub(r"\s+", " ", text).strip()


def expected_chunks(evidence, chunks):
    """Ids of the chunks of each quoted document that contain the quote, and the quotes found in no chunk."""
    normalized = chunks["text"].map(normalize)
    ids, missing = [], []
    for item in evidence:
        quote = normalize(item["quote"])
        found = chunks.loc[(chunks["doc_id"] == item["doc_id"]) & normalized.str.contains(quote, regex=False), "chunk_id"]
        if found.empty:
            missing.append(item)
        ids += [chunk_id for chunk_id in found if chunk_id not in ids]
    return ids, missing


if QUERIES_FILE.exists():
    queries = json.loads(QUERIES_FILE.read_text(encoding="utf-8"))
    for query in queries:
        query["expected_chunks"], missing = expected_chunks(query["evidence"], chunks)
        for item in missing:
            print(f"{query['id']}: cita no encontrada en {item['doc_id']}, p. {item['page']}: {item['quote'][:80]!r}")
    print(f"{len(queries)} consultas")
    display(pd.DataFrame(queries)[["id", "type", "question", "expected_chunks"]])
else:
    print("Todavía no hay consultas en data/queries.json")

10 consultas


,id,type,question,expected_chunks
0,q01,respondible,¿La obligación de guardar confidencialidad es ...,"[lfpdppp-0026, lgpdppso-0051]"
1,q02,respondible,¿En qué artículo de la LFPDPPP está el deber d...,"[lfpdppp-0026, lfpdppp-0066]"
2,q03,respondible,"Si contrato un servicio en la nube, ¿el provee...","[reglamento_lfpdppp-0060, lgpdppso-0077, inai_..."
3,q04,respondible,¿Puedo oponerme a que un banco decida si me da...,[lfpdppp-0032]
4,q05,respondible,Mi app de salud guarda el historial médico de ...,"[lfpdppp-0006, lfpdppp-0014, inai_recomendacio..."
5,q06,respondible,¿Qué le recomienda la UNESCO a México para reg...,"[unesco_ram_mexico-0112, unesco_ram_mexico-011..."
6,q07,respondible,¿Qué dicen los Principios de Chapultepec sobre...,[principios_chapultepec-0005]
7,q08,respondible,Si se aprueba la iniciativa de ley de IA del S...,[iniciativa_ley_ia_senado-0145]
8,q09,trampa,¿Qué multas establece la ley de inteligencia a...,"[unesco_ram_mexico-0031, unesco_ram_mexico-003..."
9,q10,trampa,¿Cómo presento una queja ante el INAI si una e...,"[lfpdppp-0047, lfpdppp-0049, lfpdppp-0007, lfp..."


## Embeddings

### Decisión: modelo de embeddings (BGE-M3)

Se usa **BGE-M3** (`BAAI/bge-m3`), en la propia computadora con `sentence-transformers`.

- **Multilingüe:** entrenado para recuperación en más de 100 idiomas, incluido el español.
- **No corta ningún chunk:** lee hasta 8,192 tokens. Con su tokenizador, `full_text` mide 191 tokens de mediana y 545 como máximo; un modelo con límite de 512 tokens cortaría 2 chunks.
- **Gratuito y local:** el corpus no sale de la computadora, no requiere clave de API y los resultados se pueden reproducir.
- Vectores de 1,024 dimensiones, normalizados para usar similitud coseno. Se indexa `full_text` (línea de contexto y texto).

Se descartaron multilingual-e5-large (límite de 512 tokens y prefijos obligatorios), Qwen3-Embedding-0.6B (más nuevo, pero con menos referencias) y text-embedding-3 de OpenAI (de pago, con clave de API, y depende de un modelo alojado que puede retirarse).

Los vectores se guardan en `data/embeddings/` para no recalcularlos en cada ejecución.

In [10]:
import time

import numpy as np
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "BAAI/bge-m3"
EMBEDDINGS_FILE = DATA / "embeddings" / "bge_m3.npz"

embedder = SentenceTransformer(EMBEDDING_MODEL)


def embed(texts):
    return embedder.encode(texts, batch_size=16, normalize_embeddings=True, show_progress_bar=False)


tokens = chunks["full_text"].map(lambda text: len(embedder.tokenizer(text)["input_ids"]))
print(f"Tokens por chunk: mediana {int(tokens.median())}, máximo {tokens.max()}, más de 512: {(tokens > 512).sum()}")

cached = np.load(EMBEDDINGS_FILE) if EMBEDDINGS_FILE.exists() else None
if cached is not None and cached["ids"].tolist() == chunks["chunk_id"].tolist():
    chunk_vectors = cached["vectors"]
    print(f"Embeddings leídos de {EMBEDDINGS_FILE}")
else:
    start = time.perf_counter()
    chunk_vectors = embed(chunks["full_text"].tolist())
    EMBEDDINGS_FILE.parent.mkdir(exist_ok=True)
    np.savez(EMBEDDINGS_FILE, ids=np.array(chunks["chunk_id"].tolist()), vectors=chunk_vectors)
    print(f"{len(chunk_vectors)} chunks convertidos en {time.perf_counter() - start:.0f} s")
print(f"Vectores: {chunk_vectors.shape[0]} × {chunk_vectors.shape[1]}")

C:\Projects\Maestria\PLN\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 27774.50it/s]

Tokens por chunk: mediana 191, máximo 545, más de 512: 2
Embeddings leídos de ..\data\embeddings\bge_m3.npz
Vectores: 927 × 1024


## Base de datos vectorial

### Decisión: ChromaDB

- **Sin servidor:** funciona dentro de Python y guarda la colección en disco (`data/chroma/`).
- **Texto y metadatos junto al vector:** cada chunk lleva su documento, tipo, páginas, sección y si el documento es vinculante, y se puede filtrar por ellos.
- **Escala suficiente:** 927 chunks hoy y unos 66,000 con 500 documentos; cualquier opción los busca en milisegundos, así que pesó más la sencillez. Para producción, Chroma tiene un modo cliente-servidor.
- Distancia coseno, sobre los vectores normalizados de BGE-M3.

Se descartaron Qdrant (más funciones para producción, como búsqueda híbrida nativa, pero más configuración), FAISS (no guarda texto ni metadatos, y la persistencia habría que construirla a mano) y pgvector (requiere un servidor PostgreSQL).

In [11]:
import chromadb
from chromadb.errors import NotFoundError

chroma = chromadb.PersistentClient(path=str(DATA / "chroma"))
try:
    chroma.delete_collection("chunks")  # se reconstruye en cada ejecución para que coincida con los chunks
except NotFoundError:
    pass
collection = chroma.create_collection("chunks", configuration={"hnsw": {"space": "cosine"}})
collection.add(
    ids=chunks["chunk_id"].tolist(),
    embeddings=chunk_vectors,
    documents=chunks["full_text"].tolist(),
    metadatas=[
        {
            "doc_id": chunk["doc_id"],
            "kind": chunk["kind"],
            "pages": ", ".join(map(str, chunk["pages"])),
            "heading": chunk["heading"] if isinstance(chunk["heading"], str) else "",
            "binding": MANIFEST[chunk["doc_id"]]["binding"],
        }
        for chunk in chunks.to_dict("records")
    ],
)
print(f"Chunks en la colección: {collection.count()}")

Chunks en la colección: 927


## Recuperación

Se comparan dos estrategias base, las dos sobre `full_text`:

- **BM25:** búsqueda por palabras clave. Favorece términos exactos, como siglas y números de artículo. El texto se pasa a minúsculas, sin acentos, y se separa en palabras; no se quitan palabras vacías, porque BM25 ya les da poco peso.
- **Embeddings:** búsqueda semántica con BGE-M3 en ChromaDB. Favorece preguntas dichas con otras palabras que el documento.

Después del análisis de errores se agregó una tercera, embeddings + reranker, que se explica en su propia decisión.

In [12]:
import unicodedata

from rank_bm25 import BM25Okapi


def tokenize(text):
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(character for character in text if not unicodedata.combining(character))
    return re.findall(r"\w+", text)


bm25 = BM25Okapi([tokenize(text) for text in chunks["full_text"]])
chunk_ids = chunks["chunk_id"].tolist()


def search_bm25(question, k):
    scores = bm25.get_scores(tokenize(question))
    return [chunk_ids[i] for i in np.argsort(-scores)[:k]]


def search_embeddings(question, k):
    return collection.query(query_embeddings=embed([question]), n_results=k)["ids"][0]


STRATEGIES = {"BM25": search_bm25, "Embeddings": search_embeddings}
question = queries[3]["question"]
for name, search in STRATEGIES.items():
    print(f"{name}: {search(question, 3)}")
print(f"Esperados para «{question}»: {queries[3]['expected_chunks']}")

BM25: ['lgpdppso-0065', 'iniciativa_ley_ia_senado-0023', 'inai_recomendaciones_ia-0075']


Embeddings: ['inai_recomendaciones_ia-0083', 'inai_recomendaciones_ia-0010', 'reglamento_lfpdppp-0119']
Esperados para «¿Puedo oponerme a que un banco decida si me da un crédito usando solo un algoritmo?»: ['lfpdppp-0032']


### Decisión: reranker (tercera estrategia)

Se agregó después del análisis de errores (al final del notebook). La estrategia **Embeddings + reranker** toma los 30 primeros chunks de los embeddings y los reordena con un cross-encoder, **bge-reranker-v2-m3**. Del 31 en adelante, el orden es el de los embeddings.

Por qué:

- **El chunk correcto suele estar cerca, pero fuera de los 5 primeros.** Con embeddings, el artículo 26 de la LFPDPPP (q04) quedó en la posición 27 y la tabla de recomendaciones de la UNESCO (q06) en la 11. Un reranker solo reordena candidatos, así que sirve justo cuando el chunk correcto ya está entre ellos.
- **Juzga si el pasaje responde la pregunta, no solo si trata el mismo tema.** Los embeddings comparan dos vectores calculados por separado, uno de la pregunta y otro del chunk, así que premian los chunks del mismo tema, como el prólogo y los agradecimientos de la UNESCO en q06. El cross-encoder lee la pregunta y el chunk en una sola entrada, y su atención relaciona cada palabra de la pregunta con las del chunk. Es más preciso, pero más lento y no se puede precalcular, así que se aplica solo a 30 candidatos.
- **Candidatos de los embeddings, no de BM25:** entre sus 30 primeros hay más chunks esperados (Recall@30 de 0.72, contra 0.53 de BM25), y el reranker no puede subir un chunk que no está entre los candidatos.
- **Misma familia que BGE-M3:** está construido sobre él, es multilingüe y acepta textos largos. Se usa con un límite de 1,024 tokens, más que el chunk más largo (545 tokens) junto con la pregunta.
- **Local y gratuito,** como el resto del sistema.

**Costo:** 2.3 GB de modelo y unos 13 s por consulta, porque el entorno usa PyTorch para CPU; con la GPU sería mucho más rápido.

Las otras mejoras propuestas en el análisis de errores (búsqueda híbrida, reescribir la pregunta, título corto en el índice, stemming) no se eligieron y quedan como trabajo futuro.

In [13]:
from functools import lru_cache

from sentence_transformers import CrossEncoder

RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
CANDIDATES = 30

reranker = CrossEncoder(RERANKER_MODEL, max_length=1024)
full_texts = dict(zip(chunk_ids, chunks["full_text"]))


@lru_cache(maxsize=None)
def rerank(question):
    """Ranking completo de los embeddings, con sus primeros CANDIDATES chunks reordenados por el cross-encoder."""
    ranking = search_embeddings(question, len(chunk_ids))
    candidates = ranking[:CANDIDATES]
    scores = reranker.predict([(question, full_texts[chunk_id]) for chunk_id in candidates], batch_size=8)
    return [candidates[i] for i in np.argsort(-scores, kind="stable")] + ranking[CANDIDATES:]


def search_reranker(question, k):
    return rerank(question)[:k]


for name, search in STRATEGIES.items():
    recall = np.mean([len(set(q["expected_chunks"]) & set(search(q["question"], CANDIDATES))) / len(q["expected_chunks"]) for q in queries])
    print(f"Recall@{CANDIDATES} de {name}: {recall:.2f}")

start = time.perf_counter()
for query in queries:
    rerank(query["question"])
print(f"Reranking: {(time.perf_counter() - start) / len(queries):.1f} s por consulta en CPU")

STRATEGIES["Embeddings + reranker"] = search_reranker

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   6%|▌         | 22/393 [00:00<00:01, 195.05it/s]

Loading weights:  11%|█         | 42/393 [00:00<00:02, 131.66it/s]

Loading weights:  15%|█▍        | 57/393 [00:00<00:03, 111.51it/s]

Loading weights:  18%|█▊        | 69/393 [00:00<00:03, 103.45it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 566.59it/s]

Recall@30 de BM25: 0.53


Recall@30 de Embeddings: 0.72


Reranking: 11.7 s por consulta en CPU


## Evaluación de la recuperación

- **Precision@k:** de los k primeros chunks, qué parte es esperada.
- **Recall@k:** de los chunks esperados, qué parte aparece entre los k primeros.

Se mide con k = 1, 3, 5 y 10 en las 10 consultas. Cada consulta tiene de 1 a 4 chunks esperados, así que Precision@k no puede llegar a 1 cuando k es mayor que ese número.

In [14]:
K_VALUES = [1, 3, 5, 10]
rows = []
for query in queries:
    expected = set(query["expected_chunks"])
    for name, search in STRATEGIES.items():
        retrieved = search(query["question"], max(K_VALUES))
        row = {"id": query["id"], "type": query["type"], "strategy": name, "retrieved": retrieved}
        for k in K_VALUES:
            hits = len(expected & set(retrieved[:k]))
            row[f"P@{k}"] = hits / k
            row[f"R@{k}"] = hits / len(expected)
        rows.append(row)
retrieval = pd.DataFrame(rows)

METRIC_COLUMNS = [f"{metric}@{k}" for metric in ("P", "R") for k in K_VALUES]
print("Promedio de las 10 consultas:")
display(retrieval.groupby("strategy")[METRIC_COLUMNS].mean().round(2))
print("Recall@5 por consulta:")
display(retrieval.pivot(index="id", columns="strategy", values="R@5").round(2))

Promedio de las 10 consultas:


,P@1,P@3,P@5,P@10,R@1,R@3,R@5,R@10
strategy,,,,,,,,
BM25,0.3,0.20,0.16,0.11,0.22,0.32,0.38,0.46
Embeddings,0.3,0.20,0.14,0.11,0.25,0.35,0.38,0.51
Embeddings + reranker,0.3,0.27,0.24,0.13,0.22,0.45,0.61,0.63


Recall@5 por consulta:


strategy,BM25,Embeddings,Embeddings + reranker
id,,,
q01,0.00,0.50,1.00
q02,0.50,0.50,0.50
q03,0.75,0.50,0.75
q04,0.00,0.00,1.00
q05,0.00,0.00,0.00
q06,0.00,0.00,0.33
q07,1.00,1.00,1.00
q08,1.00,1.00,1.00
q09,0.00,0.25,0.50


## Generación

### Decisión: LLM local (qwen3.5:9b en Ollama)

- **Gratuito y privado:** corre en la computadora (GPU RTX 5060 Ti de 16 GB), sin clave de API ni costo por consulta, y el corpus no sale de ella.
- **Español:** su página en Ollama declara soporte para 201 idiomas y dialectos.
- **Cabe con holgura:** ocupa 6.6 GB y deja la mitad de la GPU libre, así que responde rápido (unos 45 tokens por segundo).
- **Desventaja:** un modelo de 9 mil millones de parámetros razona peor que los grandes modelos por API en matices legales, como distinguir una ley de una iniciativa, y es menos confiable como juez de RAGAS.

Se descartaron Claude Opus 5.5, Claude Sonnet 5.5 y GPT de OpenAI (de pago, con clave de API, y el corpus sale de la computadora), gemma4:12b (más lento) y gpt-oss:20b (ocupa casi toda la GPU y se entrenó sobre todo en inglés).

Detalles de la generación:

- Ollama usa por defecto un contexto de 4,096 tokens, justo para 5 chunks más las instrucciones; se crea una variante del modelo con 16,384.
- El prompt pide responder solo con los fragmentos, citar documento y artículo o página, decir cuándo la respuesta no está, corregir premisas falsas y distinguir las leyes vigentes de los documentos no obligatorios. Cada fragmento lleva el tipo y el estatus de su documento, tomados del manifiesto.
- Temperatura 0 y sin modo de razonamiento, para respuestas reproducibles y rápidas.
- Se responde cada consulta con los 5 primeros chunks de cada estrategia. Las respuestas se guardan en `data/answers.json`; si cambian el prompt o el modelo, hay que borrar ese archivo para regenerarlas.

In [15]:
import ollama

BASE_MODEL = "qwen3.5:9b"
LLM_MODEL = "qwen3.5-9b-16k"
TOP_K = 5
ANSWERS_FILE = DATA / "answers.json"

ollama.create(model=LLM_MODEL, from_=BASE_MODEL, parameters={"num_ctx": 16384})

SYSTEM_PROMPT = """Eres un asistente sobre protección de datos personales e inteligencia artificial en México.
Responde en español usando solo los fragmentos proporcionados.
- Cita el documento y el artículo o la página de donde sale cada dato.
- Si los fragmentos no contienen la respuesta, dilo claramente y no inventes.
- Si la pregunta parte de una premisa falsa o desactualizada, corrígela.
- Distingue las leyes vigentes de las guías, informes, declaraciones e iniciativas, que no son obligatorias."""

chunk_by_id = chunks.set_index("chunk_id")
query_by_id = {query["id"]: query for query in queries}


def context_block(chunk_id):
    chunk = chunk_by_id.loc[chunk_id]
    entry = MANIFEST[chunk["doc_id"]]
    return f"[{entry['type']}; {entry['status']}; páginas {', '.join(map(str, chunk['pages']))}]\n{chunk['full_text']}"


def generate(question, retrieved):
    context = "\n\n---\n\n".join(context_block(chunk_id) for chunk_id in retrieved)
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Fragmentos:\n\n{context}\n\nPregunta: {question}"},
        ],
        think=False,
        options={"temperature": 0},
    )
    return response.message.content


previous = {}
if ANSWERS_FILE.exists():
    previous = {(item["id"], item["strategy"]): item for item in json.loads(ANSWERS_FILE.read_text(encoding="utf-8"))}

start = time.perf_counter()
answers = []
for row in retrieval.to_dict("records"):
    retrieved = row["retrieved"][:TOP_K]
    old = previous.get((row["id"], row["strategy"]))
    text = old["answer"] if old and old["retrieved"] == retrieved else generate(query_by_id[row["id"]]["question"], retrieved)
    answers.append({"id": row["id"], "type": row["type"], "strategy": row["strategy"], "retrieved": retrieved, "answer": text})
ANSWERS_FILE.write_text(json.dumps(answers, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
generated = pd.DataFrame(answers)
print(f"{len(generated)} respuestas en {time.perf_counter() - start:.0f} s")

for item in answers:
    if item["type"] == "trampa":
        print(f"\n{item['id']} · {item['strategy']} · {query_by_id[item['id']]['question']}\n{item['answer']}")

30 respuestas en 0 s

q09 · BM25 · ¿Qué multas establece la ley de inteligencia artificial que está vigente en México?
No existe una ley de inteligencia artificial vigente en México que establezca multas específicas para este sector.

Los fragmentos proporcionados indican que el documento que contiene las sanciones (incluyendo multas por uso indebido ambiental) es una **iniciativa de ley** o **propuesta**, y no una norma vigente:
*   **Documento:** Iniciativa con proyecto de decreto por el que se expide la Ley General para el Uso y Control de la Inteligencia Artificial en los Sectores Público y Privado.
*   **Estado:** Iniciativa / Propuesta (no es ley vigente).
*   **Fuente:** Páginas 39 del documento citado.

El texto especifica explícitamente que esta iniciativa "no es ley vigente". Por lo tanto, las multas descritas en el Artículo 115 de dicha propuesta (multas proporcionales al volumen de recursos mal utilizados) no están actualmente aplicables bajo una ley vigente.

Mientras tant

## Evaluación de la generación (RAGAS)

Cuatro métricas de RAGAS, cada una de 0 a 1:

- **Faithfulness:** qué parte de las afirmaciones de la respuesta está respaldada por los chunks recuperados. Mide si el modelo inventa.
- **Answer relevancy:** si la respuesta atiende la pregunta. RAGAS genera preguntas a partir de la respuesta y las compara con la original usando embeddings.
- **Context precision:** si los chunks útiles para la respuesta ideal están en las primeras posiciones.
- **Context recall:** qué parte de la respuesta ideal se encuentra en los chunks recuperados.

El juez es el mismo qwen3.5:9b, a través del endpoint compatible con OpenAI de Ollama, sin modo de razonamiento (con él, el juez gastaba sus tokens razonando y no completaba la respuesta) y con un límite de 8,192 tokens de salida (con el valor por defecto de RAGAS, 1,024, faithfulness se cortaba en 10 de las 20 respuestas, las más largas). Los embeddings de answer relevancy son de BGE-M3. Un modelo que juzga sus propias respuestas puede ser indulgente, y uno de 9B es menos confiable que un juez grande por API, así que los puntajes sirven para comparar las estrategias más que como valores absolutos.

Los puntajes se guardan en `data/ragas_scores.csv` y solo se calculan los que faltan o los de respuestas que cambiaron. Si el juez devuelve una salida mal formada, ese puntaje queda vacío y se avisa.

In [16]:
from openai import AsyncOpenAI
from ragas.embeddings import BaseRagasEmbedding
from ragas.llms import llm_factory
from ragas.metrics.collections import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness

RAGAS_FILE = DATA / "ragas_scores.csv"


class LocalEmbeddings(BaseRagasEmbedding):
    """BGE-M3, ya cargado, para answer relevancy."""

    def embed_text(self, text, **kwargs):
        return embed([text])[0].tolist()

    async def aembed_text(self, text, **kwargs):
        return self.embed_text(text)


judge = llm_factory(
    LLM_MODEL,
    provider="openai",
    client=AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama"),
    temperature=0,
    reasoning_effort="none",
    max_tokens=8192,  # con el valor por defecto (1,024) se cortaba faithfulness en las respuestas largas
)
METRICS = {
    "faithfulness": Faithfulness(llm=judge),
    "answer_relevancy": AnswerRelevancy(llm=judge, embeddings=LocalEmbeddings()),
    "context_precision": ContextPrecision(llm=judge),
    "context_recall": ContextRecall(llm=judge),
}


async def ragas_scores(query, item, names):
    contexts = [chunk_by_id.at[chunk_id, "full_text"] for chunk_id in item["retrieved"]]
    inputs = {
        "faithfulness": dict(user_input=query["question"], response=item["answer"], retrieved_contexts=contexts),
        "answer_relevancy": dict(user_input=query["question"], response=item["answer"]),
        "context_precision": dict(user_input=query["question"], reference=query["ideal_answer"], retrieved_contexts=contexts),
        "context_recall": dict(user_input=query["question"], retrieved_contexts=contexts, reference=query["ideal_answer"]),
    }
    scores = {}
    for name in names:
        try:
            scores[name] = (await METRICS[name].ascore(**inputs[name])).value
        except Exception as error:  # una salida mal formada del juez deja vacío ese puntaje
            print(f"{query['id']} · {item['strategy']} · {name}: {type(error).__name__}")
            scores[name] = None
    return scores


previous = pd.read_csv(RAGAS_FILE).set_index(["id", "strategy"]) if RAGAS_FILE.exists() else None
if previous is not None:
    # Git en Windows (core.autocrlf) cambia los saltos de línea del CSV; sin esto, ninguna respuesta coincide y se recalcula todo
    previous["answer"] = previous["answer"].str.replace("\r\n", "\n")
start = time.perf_counter()
rows = []
for item in answers:
    key = (item["id"], item["strategy"])
    scores = {}
    if previous is not None and key in previous.index and previous.at[key, "answer"] == item["answer"]:
        scores = {name: value for name, value in previous.loc[key, list(METRICS)].items() if pd.notna(value)}
    missing = [name for name in METRICS if name not in scores]
    if missing:
        scores |= await ragas_scores(query_by_id[item["id"]], item, missing)
    rows.append({"id": item["id"], "type": item["type"], "strategy": item["strategy"], **scores, "answer": item["answer"]})
ragas_results = pd.DataFrame(rows)
ragas_results.to_csv(RAGAS_FILE, index=False)
print(f"RAGAS en {time.perf_counter() - start:.0f} s")

print("Promedio de las 10 consultas:")
display(ragas_results.groupby("strategy")[list(METRICS)].mean().round(2))
print("Por consulta:")
display(ragas_results.pivot(index="id", columns="strategy", values=list(METRICS)).round(2))

RAGAS en 0 s
Promedio de las 10 consultas:


,faithfulness,answer_relevancy,context_precision,context_recall
strategy,,,,
BM25,0.92,0.60,0.50,0.64
Embeddings,0.91,0.72,0.58,0.62
Embeddings + reranker,0.89,0.75,0.65,0.67


Por consulta:


faithfulness                                  answer_relevancy  \
strategy         BM25 Embeddings Embeddings + reranker             BM25   
id                                                                        
q01              1.00       1.00                  0.87             0.79   
q02              0.75       1.00                  0.80             0.93   
q03              1.00       1.00                  0.86             0.81   
q04              0.89       0.56                  1.00             0.00   
q05              0.78       1.00                  1.00             0.00   
q06              1.00       0.94                  0.90             0.83   
q07              0.88       0.91                  0.86             0.80   
q08              1.00       0.88                  0.89             0.00   
q09              1.00       1.00                  0.75             0.93   
q10              0.96       0.83                  1.00             0.94   

                                          context_precision             \
strategy Embeddings Embeddings + reranker              BM25 Embeddings   
id                                                                       
q01            0.89                  0.95              0.00       0.37   
q02            0.86                  0.90              0.58       1.00   
q03            0.79                  0.87              1.00       0.76   
q04            0.56                  0.83              0.00       0.00   
q05            0.79                  0.75              0.00       0.37   
q06            0.93                  0.88              0.00       1.00   
q07            0.73                  0.79              1.00       0.83   
q08            0.83                  0.79              1.00       1.00   
q09            0.85                  0.72              0.92       0.48   
q10            0.00                  0.00              0.50       0.00   

                               context_recall                                   
strategy Embeddings + reranker           BM25 Embeddings Embeddings + reranker  
id                                                                              
q01                       0.25           0.67       0.67                  1.00  
q02                       0.50           1.00       0.50                  0.50  
q03                       0.89           1.00       0.75                  1.00  
q04                       0.50           0.00       0.67                  1.00  
q05                       0.70           0.00       0.00                  0.00  
q06                       0.70           0.86       1.00                  0.89  
q07                       1.00           1.00       1.00                  1.00  
q08                       1.00           1.00       1.00                  1.00  
q09                       1.00           0.33       0.67                  0.33  
q10                       0.00           0.50       0.00                  0.00

## Análisis de errores

La primera tabla muestra en qué posición del ranking completo (927 chunks) quedó cada chunk esperado. La segunda muestra los 5 primeros resultados de las consultas en las que alguna estrategia no recuperó ningún chunk esperado entre los 5 primeros. Los puntajes de RAGAS por consulta están en la tabla anterior.

In [17]:
def label(chunk_id):
    heading = chunk_by_id.at[chunk_id, "heading"]
    return f"{chunk_id} · {heading[:50] if isinstance(heading, str) else ''}"


full_rankings = {(query["id"], name): search(query["question"], len(chunks)) for query in queries for name, search in STRATEGIES.items()}

rows = []
for query in queries:
    for chunk_id in query["expected_chunks"]:
        rows.append({"id": query["id"], "chunk esperado": chunk_id} | {name: full_rankings[(query["id"], name)].index(chunk_id) + 1 for name in STRATEGIES})
print("Posición de cada chunk esperado (1 = primero):")
display(pd.DataFrame(rows).set_index(["id", "chunk esperado"]))

failed = sorted(retrieval.loc[retrieval["R@5"] == 0, "id"].unique())
top = {name: [label(chunk_id) for query_id in failed for chunk_id in full_rankings[(query_id, name)][:5]] for name in STRATEGIES}
print("Los 5 primeros resultados de las consultas con Recall@5 = 0 en alguna estrategia:")
with pd.option_context("display.max_colwidth", 100):
    display(pd.DataFrame(top, index=pd.MultiIndex.from_product([failed, range(1, 6)], names=["id", "posición"])))

Posición de cada chunk esperado (1 = primero):


BM25  Embeddings  Embeddings + reranker
id  chunk esperado                                                        
q01 lfpdppp-0026                     98          10                      5
    lgpdppso-0051                    48           3                      4
q02 lfpdppp-0026                     19           1                      2
    lfpdppp-0066                      2          37                     37
q03 reglamento_lfpdppp-0060           1           5                      1
    lgpdppso-0077                     5           3                      2
    inai_recomendaciones_ia-0049      2           8                      5
    inai_recomendaciones_ia-0052      7          30                     16
q04 lfpdppp-0032                    227          27                      2
q05 lfpdppp-0006                    164         426                    426
    lfpdppp-0014                     78          42                     42
    inai_recomendaciones_ia-0064     10          10                     12
q06 unesco_ram_mexico-0112          208          11                      5
    unesco_ram_mexico-0118          135         242                    242
    unesco_ram_mexico-0121          168         128                    128
q07 principios_chapultepec-0005       1           1                      1
q08 iniciativa_ley_ia_senado-0145     1           1                      1
q09 unesco_ram_mexico-0031           11          52                     52
    unesco_ram_mexico-0032          505         122                    122
    iniciativa_ley_ia_senado-0142   273           3                      3
    iniciativa_ley_ia_senado-0146    10           8                      2
q10 lfpdppp-0047                      4          18                     11
    lfpdppp-0049                      2          19                      9
    lfpdppp-0007                    459         593                    593
    lfpdppp-0075                    655         528                    528

Los 5 primeros resultados de las consultas con Recall@5 = 0 en alguna estrategia:


BM25  \
id  posición                                                                                        
q01 1           inai_recomendaciones_ia-0071 · ¿CUÁL ES LA RELACIÓN ENTRE LA INTELIGENCIA ARTIFIC   
    2                   inai_recomendaciones_ia-0052 · II. Cuente con mecanismos, al menos, para:   
    3                 unesco_ram_mexico-0048 · DATOS ABIERTOS, ACCESO A LA INFORMACIÓN Y TRANSPAR   
    4          iniciativa_ley_ia_senado-0004 · Capacitación Educacional en el Uso y Aplicación de   
    5         inai_recomendaciones_ia-0060-2 · ¿CUÁL ES LA RELACIÓN ENTRE LA INTELIGENCIA ARTIFIC   
q04 1                          lgpdppso-0065 · Del Ejercicio de los Derechos de Acceso, Rectifica   
    2          iniciativa_ley_ia_senado-0023 · Artículo 8. Para los efectos de la presente ley, s   
    3                                     inai_recomendaciones_ia-0075 · PRINCIPIO DE INFORMACIÓN   
    4                                     unesco_ram_mexico-0101 · INFRAESTRUCTURA Y CONECTIVIDAD   
    5                                             lgpdppso-0073 · Responsable y Persona Encargada   
q05 1                      iniciativa_ley_ia_senado-0090 · Capítulo 111 De los Servicios de Salud   
    2                      iniciativa_ley_ia_senado-0087 · Capítulo 111 De los Servicios de Salud   
    3                      iniciativa_ley_ia_senado-0089 · Capítulo 111 De los Servicios de Salud   
    4                      iniciativa_ley_ia_senado-0088 · Capítulo 111 De los Servicios de Salud   
    5                                          iniciativa_ley_ia_senado-0009 · Sector de la Salud   
q06 1                                                    unesco_ram_mexico-0003 · AGRADECIMIENTOS   
    2          iniciativa_ley_ia_senado-0047 · TÍTULO SEGUNDO DE LOS ÓRGANISMOS PÚBLICOS DE CONTR   
    3                                                    unesco_ram_mexico-0002 · AGRADECIMIENTOS   
    4                 unesco_ram_mexico-0139 · 4.2 ÉTICA, RESPONSABILIDAD E INCLUSIÓN. IA DE INTE   
    5                          inai_recomendaciones_ia-0013 · ¿QUÉ ES LA INTELIGENCIA ARTIFICIAL?   
q09 1                            iniciativa_ley_ia_senado-0148 · TÍTULO SEXTO MARCO SANCIONATORIO   
    2                                   unesco_ram_mexico-0037 · PRIVACIDAD Y PROTECCIÓN DE DATOS   
    3                   iniciativa_ley_ia_senado-0020 · Artículo 3. Son fines de la presente Ley:   
    4                                       iniciativa_ley_ia_senado-0003 · EXPOSICIÓN DE MOTIVOS   
    5                              unesco_ram_mexico-0052 · DEBIDO PROCESO Y RENDICIÓN DE CUENTAS   
q10 1                reglamento_lfpdppp-0040 · Procedimiento para la autorización de medidas comp   
    2                           lfpdppp-0049 · Capítulo VIII Del Procedimiento de Protección de D   
    3                           lfpdppp-0058 · Capítulo VIII Del Procedimiento de Protección de D   
    4                           lfpdppp-0047 · Capítulo VIII Del Procedimiento de Protección de D   
    5                                                              lgpdppso-0046 · De los Deberes   

                                                                                     Embeddings  \
id  posición                                                                                      
q01 1         inai_recomendaciones_ia-0071 · ¿CUÁL ES LA RELACIÓN ENTRE LA INTELIGENCIA ARTIFIC   
    2                        lgpdppso-0139 · Del Recurso de Revisión en Materia de Seguridad Na   
    3                                                            lgpdppso-0051 · De los Deberes   
    4                                         lgpdppso-0144 · Del Procedimiento de Verificación   
    5                                 unesco_ram_mexico-0037 · PRIVACIDAD Y PROTECCIÓN DE DATOS   
q04 1                                          inai_recomendaciones_ia-0083 · MEJORES PRÁCTICAS   
    2                                               inai_recomendaci

### Recuperación

BM25 y embeddings empatan en promedio (Recall@5 de 0.38), pero fallan en consultas distintas. BM25 gana en q03 y q10, cuyas preguntas comparten palabras con los pasajes («nube», «confidencialidad», «solicitud»). Los embeddings ganan en q01 y q09, y acercan los pasajes escritos con otras palabras (en q04, de la posición 227 a la 27). Las causas de los fallos:

1. **La pregunta y la ley usan palabras distintas (q01, q04, q05).** Las preguntas dicen «banco», «algoritmo», «app de salud», «empresas» o «gobierno»; la ley dice «tratamiento automatizado», «sin intervención humana», «datos personales sensibles», «responsable» o «sujetos obligados». BM25 deja el artículo 26 de la LFPDPPP (q04) en la posición 227, y los embeddings lo suben a la 27, todavía fuera de los 10 primeros. Es la consecuencia aceptada de escribir las preguntas como las haría un usuario.
2. **BM25 no relaciona las formas de una palabra (q06).** «Recomienda» y «regular» no coinciden con «recomendaciones» ni con «regulación». Las palabras que sí coinciden no distinguen entre chunks: «México» e «inteligencia artificial» están en la línea de contexto de todos los chunks del informe, y «UNESCO» aparece sobre todo en los agradecimientos (4 veces en cada uno), que quedan primeros. Los tres chunks esperados, que no mencionan a la UNESCO, quedan entre las posiciones 135 y 208.
3. **Una palabra del tema domina (q05 con BM25).** «Salud» está en el encabezado «Capítulo 111 De los Servicios de Salud» de la iniciativa (el «111» es un «III» mal leído en el texto del PDF), que se repite en la línea de contexto de cada chunk de ese capítulo: los 4 primeros resultados son de ahí, y el quinto es otra sección de la iniciativa sobre salud. La definición de datos sensibles (lfpdppp-0006) menciona la salud una sola vez, dentro de una lista, y queda en la posición 164 (426 con embeddings).
4. **Chunks generales o casi vacíos (q06 con embeddings).** Ante una pregunta amplia, los embeddings prefieren los chunks que resumen todo el documento: el prólogo, los agradecimientos y el diagnóstico general del informe de la UNESCO. En tercer lugar queda inai_recomendaciones_ia-0085, cuyo texto es solo «i n a i . o r g . m x»: su vector es casi el de su línea de contexto, el título de la guía del INAI («Recomendaciones para el tratamiento de datos personales derivado del uso de la inteligencia artificial»). La tabla de recomendaciones de la UNESCO (unesco_ram_mexico-0112) queda en la posición 11.
5. **La palabra de la trampa atrae pasajes que confirman la premisa (q10 con embeddings).** Los 5 primeros resultados mencionan al INAI, y cuatro son del informe de la UNESCO de 2024, que lo describe como el organismo garante. Los pasajes que corrigen la premisa (la definición de la Secretaría en el artículo 2 y el transitorio Cuarto de la LFPDPPP) no comparten palabras con la pregunta y quedan después de la posición 450 con BM25 y con embeddings. Una búsqueda por similitud encuentra lo que se parece a la pregunta, no lo que la contradice.
6. **Pasajes relevantes sin anotar (q01, q09).** Algunos chunks recuperados también responden la pregunta, pero no están entre los esperados, así que Precision@k subestima esos casos: unesco_ram_mexico-0054 («en México no existe ninguna ley o política…» sobre la IA en la justicia) en q09, e inai_recomendaciones_ia-0071, la tabla de la guía del INAI sobre el deber de confidencialidad, primera con BM25 y con embeddings en q01.

### Generación

1. **Fiel al fragmento, pero con la numeración anterior (q01 con BM25).** La respuesta da como fundamento los artículos 21 de la LFPDPPP y 42 de la LGPDPPSO, tomados de la tabla de la guía del INAI, que usa la numeración anterior a las leyes de 2025 (hoy son el 20 y el 36). El encabezado del fragmento decía «anterior a la ley de 2025», y aun así los presentó como vigentes. Como no se recuperó la LGPDPPSO, concluyó además que al gobierno lo rigen decretos de datos abiertos. Faithfulness fue 1.0: la métrica revisa que la respuesta esté en el contexto, no que el contexto esté vigente.
2. **Sigue la premisa aunque el fragmento la contradiga (q10 con BM25).** Se recuperó el procedimiento correcto y la respuesta copia que la solicitud se presenta «ante la Secretaría», pero lo presenta todo como una queja ante el INAI, y donde el artículo dice «la Secretaría dará vista al citado responsable» escribe «El INAI dará vista». La instrucción de corregir premisas funcionó en q09, donde los fragmentos lo dicen explícitamente (la etiqueta «propuesta; no es ley vigente»), y no aquí, donde había que deducirlo. Faithfulness fue 0.96 y no lo detectó. Con embeddings no se recuperó el procedimiento, y la respuesta «corrige» la pregunta con la descripción de la UNESCO de 2024, que refuerza que el INAI es el órgano garante.
3. **Fundamento equivocado cuando falta el artículo correcto (q04 y q05 con embeddings).** En q04 responde que sí, pero con el derecho de oposición general (artículo 109 del Reglamento), no con el artículo 26, fracción II, de la LFPDPPP, que trata justo las decisiones automatizadas. En q05 está el error más grave: sin la definición de datos sensibles ni el artículo 8, dice que la app podría no necesitar consentimiento por la excepción de atención médica del artículo 9, lo contrario de la respuesta ideal (consentimiento expreso y por escrito). Aun así, su faithfulness es 1.0, porque cada afirmación sale de algún fragmento.
4. **Abstención correcta (q04 y q05 con BM25).** Sin los pasajes, el modelo dijo que los fragmentos no permiten responder, en lugar de inventar, aunque en q04 citó la LGPDPPSO, que es para el sector público y no aplica a un banco.
5. **Generaliza de los fragmentos al documento (q06).** Las dos respuestas afirman que la UNESCO no hace recomendaciones específicas para México; el informe tiene una sección de recomendaciones principales, pero no se recuperó.

### Qué tan confiable es RAGAS aquí

- **Answer relevancy castiga las salvedades.** Da 0 cuando detecta una respuesta evasiva. q08 con BM25 es completa y correcta, pero empieza con «no puedo confirmar el castigo… como una ley vigente», la salvedad que pide el prompt, y recibe 0. Los ceros explican la diferencia entre estrategias: BM25 tiene tres (q04, q05 y q08) y embeddings uno (q10); sin ellos, BM25 promedia 0.86 y embeddings 0.80.
- **El juez local es indulgente con el contexto.** En q06 con embeddings, context precision y context recall son 1.0 con el prólogo y los agradecimientos, aunque ninguna de las recomendaciones de la respuesta ideal está en ellos (Recall@5 anotado: 0). Para comparar la recuperación valen más Precision@k y Recall@k, anotados a mano.
- **El juez no da lo mismo en cada corrida.** Al recalcular los 120 puntajes con las mismas respuestas, cambiaron 15, en 13 de las 30 respuestas: 0.17 en promedio y hasta 0.5. Los promedios por estrategia se movieron hasta 0.05, tanto como algunas diferencias entre estrategias (context precision: 0.58 con embeddings y 0.65 con el reranker). Por eso las diferencias pequeñas de RAGAS no se interpretan.
- **Faithfulness alta no significa respuesta correcta.** q01 y q10 con BM25, q05 con embeddings y q06 con BM25 y con embeddings tienen 0.94 o más y son incorrectas: la métrica mide si la respuesta se apoya en los chunks recuperados, no si son los correctos ni si están vigentes.

### Resultado del reranker

El reranker se eligió de estas propuestas y se agregó como tercera estrategia (ver «Decisión: reranker»). Recall@5 sube de 0.38 a 0.61 y Precision@5 de 0.14 a 0.24, a unos 13 s por consulta en CPU.

- **Lo que corrigió:** q04 (el artículo 26 de la LFPDPPP sube de la posición 27 a la 2), q01 (los dos artículos esperados entran a los 5 primeros) y q06 (la tabla de recomendaciones de la UNESCO sube de la 11 a la 5). En las tres, la respuesta pasó a ser correcta: q04 cita el artículo 26 y su excepción, q01 los artículos 20 de la LFPDPPP y 36 de la LGPDPPSO, y q06 las recomendaciones del informe.
- **Lo que no corrigió:** los pasajes que no estaban entre los 30 candidatos de los embeddings. En q05, solo la guía del INAI estaba entre ellos, y quedó en la posición 12; la respuesta ya no dice que la app podría no necesitar consentimiento, pero sin la definición de datos sensibles concluye que el consentimiento expreso es solo una buena práctica. En q10, el procedimiento sube a las posiciones 9 y 11, todavía fuera de los 5 primeros, y la respuesta dice que no encuentra el procedimiento y repite que el INAI es el organismo garante.
- **RAGAS:** answer relevancy sube a 0.75, context precision a 0.65 y context recall a 0.67; faithfulness queda en 0.89.

### Qué lo mejoraría

Propuestas; solo el reranker se implementó:

1. **Búsqueda híbrida:** combinar BM25 y embeddings con Reciprocal Rank Fusion, porque fallan en consultas distintas.
2. **Reranker sobre los 30 primeros**, por ejemplo bge-reranker-v2-m3, un cross-encoder que lee la pregunta y el chunk juntos: el artículo 26 de q04 está en la posición 27 y la tabla de la UNESCO de q06 en la 11. **Implementado** (ver «Resultado del reranker»).
3. **Ajustar la línea de contexto:** quitar el título del documento de lo que se indexa (dejar el encabezado de sección) y descartar chunks casi vacíos como inai_recomendaciones_ia-0085.
4. **Stemming en español para BM25**, para que «recomienda» coincida con «recomendaciones».
5. **Reescribir la pregunta con el LLM** en vocabulario legal antes de buscar (q04, q05), a costa de una llamada más.
6. **Más metadatos en el encabezado de cada fragmento**, como el estatus que ya ayudó en q09; por ejemplo, la autoridad que aplica cada ley (la Secretaría Anticorrupción y Buen Gobierno en la LFPDPPP).
7. **Un juez más grande por API** para RAGAS.